# extract

> what a document *is*, the fields inside it, and an answer over the whole of it

In [ ]:
#| default_exp extract

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Retrieval answers *where* something is. This module answers two questions it cannot: **what kind of
thing** each document in the vault is, and **what fields** are inside it — a vault that knows which
of its documents are invoices can hand you their totals as a table.

Three legs, cheapest first, and each says which one answered:

|leg|needs|what it does|
|---|---|---|
|regex signals|nothing|money, dates, references, tables, code — the evidence a type is scored on|
|spaCy|`vishalakshi[nlp]`|named entities: `ORG`, `PERSON`, `GPE`, `PRODUCT`|
|a small LLM|`rishi`|the label when the cues cannot decide, and every structured extraction|

The order is the design. Typing ten thousand documents through an LLM is hours of compute to answer
a question a cue table answers for most of them — so the model is spent only where the table knows
it is guessing.

In [ ]:
#| export
import json, re, warnings
from collections import Counter
from dataclasses import dataclass, fields, is_dataclass, make_dataclass, asdict
from typing import get_origin
from fastcore.all import AttrDict, L, patch
from vishalakshi.core import Vault
from vishalakshi.ask import VAULT_SP, cited, resolve_model, split_reasoning   # also patches Vault.chat

## Signals: what a document contains

`signals` counts the evidence, and is the one call that both legs of categorisation share.

In [ ]:
#| export
SIGNALS = dict(
    # what a document *contains*, as cheaply as a regex can tell — the evidence a type is scored on
    money    = r'[$€£¥₹]\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*\.\d{2}\s?(?:usd|eur|gbp|inr|jpy|cad|aud)\b'
               r'|\b(?:usd|eur|gbp|inr|jpy|cad|aud)\s?\d[\d,]*',
    date     = r'\b\d{4}-\d{2}-\d{2}\b|\b\d{1,2}[/.]\d{1,2}[/.]\d{2,4}\b'
               r'|\b\d{1,2} (?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{2,4}\b'
               r'|\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]* \d{1,2},? \d{4}\b',
    time     = r'\b\d{1,2}:\d{2}(?::\d{2})?\s?(?:am|pm)?\b',
    ref      = r'\b(?:invoice|inv|order|p\.?o|ref(?:erence)?|receipt|bill|account|sku|part|item)\s*'
               r'(?:no\.?|number|id)?\s*[:#]\s*[a-z0-9][a-z0-9\-/]{2,}\b',
    qty      = r'\b(?:qty|quantity|units?|pcs?|nos?)\b\s*[:.]?\s*\d+|\b\d+\s?(?:x|×)\s?\d',
    percent  = r'\b\d{1,3}(?:\.\d+)?\s?%',
    tax      = r'\b(?:vat|gst|hsn|sac|tin|ein|abn|sales tax|tax id|withholding)\b',
    email    = r'\b[\w.+-]+@[\w-]+\.[\w.]{2,}\b',
    url      = r'https?://\S+',
    citation = r'\[\d+\]|\bet al\.|\bdoi:|\barxiv:',
    code     = r'^\s*(?:def|class|function|const|let|var|import|from|package|#include)\s+\w',
    table    = r'^\s*\|.+\|\s*$',
    heading  = r'^#{1,6} \S',
    blank    = r'_{4,}|\[ ?\]',
)
_SIG = {k: re.compile(v, re.I|re.M) for k, v in SIGNALS.items()}
# spaCy over a whole book is slow, and the head of a document is where its type is decided
NER_CHARS, _NLP = 20000, {}

def nlp(model:str='en_core_web_sm', quiet:bool=False):
    """A cached spaCy pipeline, or `None` when spaCy or its model is not installed.

    Absent is a normal state, not an error: the parser and the lemmatizer are excluded because
    nothing here needs them, and everything that uses this degrades to regex signals and says so —
    the same bargain `mk_encoder` strikes with `hash_embed`."""
    if model in _NLP: return _NLP[model]
    try:
        import spacy
        _NLP[model] = spacy.load(model, exclude=['parser', 'lemmatizer'])
    except Exception as e:
        if not quiet: warnings.warn(
            f'no spaCy pipeline {model!r} ({type(e).__name__}) — named entities will come from regex '
            f"signals only. For real NER: pip install 'vishalakshi[nlp]' && python -m spacy download {model}")
        _NLP[model] = None
    return _NLP[model]

def signals(text:str,            # the document text
            ner:bool=True,       # run spaCy over the head of it, if a pipeline is installed
            model:str='en_core_web_sm',
            limit:int=20,        # distinct entities kept
) -> AttrDict:
    """Countable evidence in one document: `counts` per signal, plus named entities.

    Regex answers always; spaCy answers when installed, and reaches the labels a regex cannot —
    `ORG`, `PERSON`, `GPE`, `PRODUCT`. Its labels are merged into `counts`, so a caller asks "is
    there money in here" once and does not care which leg found it, and kept *separately* in
    `labels`, so a scorer that wants to know what spaCy specifically saw is not handed a regex hit
    wearing the same name. `method` says which legs ran, because a score is only comparable against
    another score from the same legs."""
    counts = {k: len(p.findall(text or '')) for k, p in _SIG.items()}
    labels, ents, d = {}, L(), (nlp(model) if ner else None)
    if d is not None:
        seen = {}
        for e in d((text or '')[:NER_CHARS]).ents:
            k = (e.label_, re.sub(r'\s+', ' ', e.text.strip())[:60])
            if not k[1]: continue
            seen[k] = seen.get(k, 0) + 1
            labels[e.label_.lower()] = labels.get(e.label_.lower(), 0) + 1
        counts = {k: counts.get(k, 0) + labels.get(k, 0) for k in (*counts, *labels)}
        ents = L(AttrDict(label=l, text=t, n=n) for (l, t), n in
                 sorted(seen.items(), key=lambda kv: -kv[1])[:limit])
    else:
        ents = L(AttrDict(label=k.upper(), text=re.sub(r'\s+', ' ', m.strip())[:60], n=1)
                 for k in ('money', 'date', 'ref', 'email', 'url')
                 for m in _SIG[k].findall(text or '')[:4] if isinstance(m, str))[:limit]
    return AttrDict(counts={k: v for k, v in counts.items() if v}, labels=labels,
                    ents=ents, method='spacy+regex' if d is not None else 'regex')

## Doctypes: what a document is

A cue table, scored. `cue_scores` is a pure function of the text, which is what makes it testable —
and `guess_type` reports whether the winner is worth trusting.

In [ ]:
#| export
DOCTYPES = {
    # label: the cue phrases that are evidence for it, the regex signals it needs, and the spaCy
    # entity labels it expects. Each leg is scored as a *fraction* matched, never a count, so a long
    # document cannot out-score a short one on the same evidence.
    'invoice': dict(needs=('money',), ents=('MONEY', 'ORG', 'DATE'), cues=(
        r'\b(?:tax )?invoice\b', r'\b(?:amount|balance|total) due\b', r'\bbill(?:ed)? to\b|\bremit\b',
        r'\bpayment terms?\b|\bnet \d{1,3}\b', r'\bsub-?total\b', r'\b(?:vat|gst|sales tax)\b')),
    'receipt': dict(needs=('money',), ents=('MONEY', 'ORG'), cues=(
        r'\breceipt\b', r'\bthank you for your (?:order|purchase|payment)\b',
        r'\bcard\b[^\n]{0,16}\bending\b|\bcash tendered\b|\bchange due\b',
        r'\btransaction (?:id|no)\b|\bauth(?:orisation|orization) code\b',
        r'\bpaid\b|\bpayment received\b', r'\bmerchant\b|\bstore #\s?\d+\b')),
    'purchase_order': dict(needs=('ref',), ents=('ORG', 'DATE'), cues=(
        r'\bpurchase order\b|\bp\.?o\.?\s*(?:no|number|#)', r'\bship(?:[ -]?to)?\b',
        r'\bdelivery date\b|\brequested delivery\b', r'\bvendor\b|\bsupplier\b',
        r'\brequisition\b', r'\bunit price\b')),
    'quote': dict(needs=('money',), ents=('MONEY', 'ORG'), cues=(
        r'\bquotation\b|\bquote\s*(?:no|number|#)|\bestimate\b', r'\bvalid (?:until|for|through)\b',
        r'\bunit price\b', r'\bterms and conditions\b',
        r'\bwe are pleased to (?:quote|offer)\b|\bno obligation\b')),
    'catalogue': dict(needs=('money',), ents=('PRODUCT', 'MONEY'), cues=(
        r'\bcatalog(?:ue)?\b|\bprice list\b|\bproduct list\b',
        r'\bsku\b|\bmodel\s*(?:no|number)\b|\bpart\s*(?:no|number)\b',
        r'\b(?:in|out of) stock\b|\bavailability\b', r'\bper (?:unit|pack|case|kg|litre|liter)\b',
        r'\bspecifications?\b|\bdimensions\b', r'\badd to (?:cart|basket)\b')),
    'contract': dict(needs=(), ents=('ORG', 'DATE', 'LAW'), cues=(
        r'\bagreement\b|\bcontract\b', r'\bparties\b|\bby and between\b',
        r'\bhereby\b|\bwhereas\b|\bhereinafter\b', r'\bshall\b',
        r'\bgoverning law\b|\bjurisdiction\b|\btermination\b',
        r'\bconfidential(?:ity)?\b|\bindemnif|\bliability\b')),
    'resume': dict(needs=('email',), ents=('PERSON', 'ORG', 'DATE'), cues=(
        r'\b(?:curriculum vitae|resum[eé])\b',
        r'\bwork experience\b|\bemployment history\b|\bprofessional experience\b',
        r'\beducation\b', r'\bskills\b', r'\bcertification',
        r'\breferences available\b|linkedin\.com/in/')),
    'paper': dict(needs=('citation',), ents=('ORG', 'PERSON'), cues=(
        r'\babstract\b', r'\bintroduction\b', r'\brelated work\b|\bmethodology\b|\bexperiments?\b',
        r'\breferences\b|\bbibliography\b', r'\bwe (?:propose|present|show|evaluate)\b',
        r'\bdoi:|\barxiv:')),
    'report': dict(needs=(), ents=('MONEY', 'PERCENT', 'ORG'), cues=(
        r'\bexecutive summary\b', r'\bfindings\b|\bconclusions?\b|\brecommendations?\b',
        r'\bq[1-4] \d{4}\b|\bfiscal year\b|\bfy\d{2}\b|\bquarter\b',
        r'\btable \d+\b|\bfigure \d+\b|\bappendix\b',
        r'\byear[- ]on[- ]year\b|\brevenue\b|\bmargin\b')),
    'documentation': dict(needs=('heading',), ents=(), cues=(
        r'\binstallation\b|\bgetting started\b|\bquick ?start\b',
        r'\busage\b|\bexamples?\b|\bapi reference\b', r'\bparameters?\b|\breturns\b|\barguments?\b',
        r'\bpip install\b|\bnpm install\b|\bdocker run\b', r'\bconfiguration\b|\bsee also\b',
        r'^```')),
    'code': dict(needs=('code',), ents=(), cues=(
        r'^\s*(?:def|class)\s+\w+|^\s*(?:function|const|let|var)\s+\w+',
        r'^\s*(?:import|from|#include|package|use)\s+\w', r'\breturn\b',
        r'^\s*(?://|#|/\*)', r'\b(?:if|for|while)\s*\(', r'\b(?:public|private|static|async)\b')),
    'email': dict(needs=('email',), ents=('PERSON',), cues=(
        r'^(?:from|to|cc|bcc|subject|sent):', r'^\s*(?:dear|hi|hello)\b',
        r'\b(?:best|kind) regards\b|\bsincerely\b|\bthanks,\b',
        r'\bforwarded message\b|\bwrote:\s*$', r'\bunsubscribe\b')),
    'meeting_notes': dict(needs=(), ents=('PERSON', 'DATE'), cues=(
        r'\b(?:meeting|call) notes\b|\bminutes\b', r'\battendees\b|\bparticipants\b|\bpresent\b',
        r'\bagenda\b', r'\baction items?\b|\bnext steps\b|\bfollow[- ]ups?\b',
        r'\bdecided\b|\bdecisions?\b|\bowner\b|\bdue by\b')),
    'form': dict(needs=('blank',), ents=(), cues=(
        r'\bplease (?:complete|fill|print|sign)\b', r'\bfor office use only\b',
        r'\bapplicant\b|\bapplication (?:form|for)\b', r'\bsignature\b|\bdate signed\b',
        r'\b(?:full name|surname|given names?|date of birth|dob)\b', r'_{4,}')),
    'transcript': dict(needs=('time',), ents=('PERSON',), cues=(
        r'\btranscript\b', r'^\s*\[?\d{1,2}:\d{2}', r'^\s*(?:speaker \d|[A-Z][a-z]+\s?[A-Z]?[a-z]*):',
        r'\b(?:um|uh|you know|i mean)\b', r'\bwelcome (?:back|to)\b|\bthanks for (?:watching|listening)\b')),
    'article': dict(needs=(), ents=('PERSON', 'ORG', 'DATE'), cues=(
        r'\bpublished\b|\bposted (?:on|by)\b', r'\bread more\b|\bshare this\b|\bcomments?\b',
        r'\bsubscribe\b|\bnewsletter\b', r'\baccording to\b', r'\btags?:|\bcategor(?:y|ies):')),
}
_CUES = {l: [re.compile(c, re.I|re.M) for c in d['cues']] for l, d in DOCTYPES.items()}

# What acquired a document is evidence about what it is, and better evidence than any cue phrase:
# a YouTube ingest *is* a transcript. Only the certain cases are listed.
KIND_HINT = dict(youtube='transcript', arxiv='paper', code='code')
MIN_SCORE, MIN_MARGIN, KIND_BONUS = 0.4, 0.12, 0.2

def cue_scores(text:str, sig=None) -> dict:
    """Every doctype scored against `text`, best first — the categorisation no model is needed for.

    Three legs: the fraction of cue phrases that matched, whether the signals the type *needs* are
    present at all, and the fraction of expected spaCy entity labels seen — that last read off
    `labels` rather than `counts`, so `MONEY` is credited to it only when spaCy itself found money.
    The entity leg is dropped and its weight redistributed when spaCy is absent, which keeps scores
    on a machine without it comparable to scores on one with it."""
    sig = sig if sig is not None else signals(text)
    ner, out = sig.method != 'regex', {}
    for lbl, d in DOCTYPES.items():
        cues = [p for p in _CUES[lbl] if p.search(text or '')]
        need = [s for s in d['needs'] if sig.counts.get(s)]
        ents = [e for e in d['ents'] if sig.labels.get(e.lower())] if ner else []
        legs = [(0.6, len(cues)/len(_CUES[lbl]))]
        if d['needs']: legs.append((0.25, len(need)/len(d['needs'])))
        if ner and d['ents']: legs.append((0.15, len(ents)/len(d['ents'])))
        w = sum(x for x, _ in legs)
        out[lbl] = round(sum(x*f for x, f in legs)/w, 3)
    return dict(sorted(out.items(), key=lambda kv: -kv[1]))

def guess_type(text:str,             # the document text
               sig=None,             # a signals() result, computed if not given
               kind:str=None,        # the vault kind that acquired it, if known
               min_score:float=MIN_SCORE,    # below this the cues have not decided
               min_margin:float=MIN_MARGIN,  # runner-up this close means they have not either
) -> AttrDict:
    """The best doctype for `text` from cues alone, and whether that guess is worth trusting.

    `decisive` is the whole point: a cue table is right often and cheaply, and *knows when it is
    guessing* — a clear winner needs no model, and a two-way tie is exactly the case worth spending
    one on. `Vault.categorize` reads it to decide whether to call an LLM at all."""
    sig = sig if sig is not None else signals(text)
    sc = cue_scores(text, sig)
    if kind in KIND_HINT: sc[KIND_HINT[kind]] = round(min(1.0, sc[KIND_HINT[kind]] + KIND_BONUS), 3)
    ranked = sorted(sc.items(), key=lambda kv: -kv[1])
    (top, best), (_, second) = ranked[0], (ranked[1] if len(ranked) > 1 else ('', 0.0))
    return AttrDict(doctype=top if best else 'other', score=best, margin=round(best-second, 3),
                    decisive=best >= min_score and best-second >= min_margin,
                    scores=dict(ranked[:5]), method=sig.method)

`categorize` puts the two together, and writes the verdict into the document's `meta` so the vault
remembers it.

In [ ]:
#| export
TYPE_SP = """You label one document with what kind of thing it is.

Judge the document as a whole, by its purpose, not by a word that happens to appear in it: a paper
about invoicing is a paper. Reply with exactly one label from the list and nothing else."""

@patch
def categorize(self:Vault,
               ref,                # doc_id, source, title, a path on disk, or a loaded `document()`
               model:str=None,     # a MODELS alias or a full id, for the LLM leg
               llm:bool=None,      # None -> only when the cues cannot decide; True/False -> always/never
               labels:str=None,    # comma-separated labels to choose from; None -> DOCTYPES
               max_chars:int=6000, # chars of the document the model and the cues see
               ner:bool=True,      # run spaCy, if installed
               save:bool=True,     # write the verdict into the document's meta
) -> AttrDict:
    """What kind of document this is: invoice, catalogue, contract, paper, transcript, code…

    Cues first, a model only if they cannot decide. That order is the design, not an optimisation:
    typing a vault of ten thousand documents through an LLM is hours of compute to answer a question
    a regex table answers for most of them, and the cases the table gets wrong are the ones where it
    scores two types nearly the same — which is precisely when `llm=None` spends a model call.
    `by` records which leg decided, so a vault's types can be audited and re-run selectively. A
    `ref` that is already a `document()` is used as it stands, which is how `extract` types what it
    has just read without reassembling it twice."""
    d = (ref if isinstance(ref, dict) and 'text' in ref
         else self.document(ref, max_chars=max(max_chars, 4000)))
    txt = (d.text or '')[:max_chars]
    if not txt.strip():
        return AttrDict(doc_id=d.doc_id, title=d.title, doctype=None, skipped='no text to judge')
    sig = signals(txt, ner=ner)
    g = guess_type(txt, sig, kind=d.kind)
    dt, by = g.doctype, f'cues ({g.method})'
    if llm is True or (llm is None and not g.decisive):
        lbls = L(labels.split(',') if isinstance(labels, str) else labels or list(DOCTYPES)).map(str.strip)
        try:
            said = self.chat(model=model).classify(f'{d.title}\n\n{txt}', list(lbls)+['other'], sp=TYPE_SP)
            if said in lbls or said == 'other': dt, by = said, f'llm ({resolve_model(model)[0]})'
            else: by = f'cues ({g.method}); llm answered {said[:40]!r}, not a label'
        except Exception as e:
            # `llm=None` asked for a model *if one can be had*. There may be no rishi, no weights and
            # no network, and a corpus 90% typed by cues alone is worth far more than a traceback —
            # so the cue verdict stands, and says why. `llm=True` was an instruction, and raises.
            if llm is True: raise
            by = f'cues ({g.method}); no model available ({type(e).__name__})'
    res = AttrDict(doc_id=d.doc_id, title=d.title, kind=d.kind, doctype=dt, score=g.score,
                   margin=g.margin, decisive=g.decisive, by=by, scores=g.scores,
                   signals=sig.counts, ents=sig.ents, saved=False)
    if save and d.doc_id:
        self.set_meta(d.doc_id, doctype=dt, doctype_by=by, doctype_score=g.score)
        res.saved = True
    return res

@patch
def categorize_all(self:Vault,
                   kind:str=None,     # restrict to one or more KINDS
                   force:bool=False,  # re-type documents that already carry a doctype
                   limit:int=None,    # stop after this many
                   **kw               # forwarded to categorize (model=, llm=, ner=, max_chars=)
) -> AttrDict:
    """Type every document in the vault that is not typed yet, and report the shape of the corpus.

    One failure is recorded and stepped over rather than raised: a run over a whole vault must not
    be lost to a single unreadable document. `force=False` makes this cheap to re-run after an
    ingest — only what arrived since is looked at."""
    docs = self.sources(kind)
    if not force: docs = docs.filter(lambda r: not (r['meta'] or {}).get('doctype'))
    out = L()
    for r in docs[:limit]:
        try: out.append(self.categorize(r['id'], **kw))
        except Exception as e:
            out.append(AttrDict(doc_id=r['id'], title=r['title'], doctype=None,
                                error=f'{type(e).__name__}: {str(e)[:200]}'))
    return AttrDict(n=len(out), by_type=dict(Counter(o.doctype for o in out).most_common()),
                    errors=out.filter(lambda o: o.get('error')), results=out)

@patch
def doctypes(self:Vault, kind:str=None) -> dict:
    'How many documents of each type the vault holds — `untyped` counts the ones never categorised.'
    c = Counter((r['meta'] or {}).get('doctype') or 'untyped' for r in self.sources(kind))
    return dict(c.most_common())

@patch
def of_type(self:Vault, doctype:str, kind:str=None) -> L:
    'Every document categorised as `doctype`, newest first.'
    return self.sources(kind).filter(lambda r: (r['meta'] or {}).get('doctype') == doctype)

@patch
def ner(self:Vault, ref:str, model:str='en_core_web_sm', limit:int=40, max_chars:int=20000) -> AttrDict:
    """Named entities in one document, through spaCy — who, where, how much, when.

    Not the same thing as `connect()`, and not a replacement for it: that builds one graph over the
    whole corpus so sections can reach each other, while this reads a single document and labels
    what it names. Without spaCy the labels come from the regex signals instead, and `method` says so."""
    d = self.document(ref, max_chars=max_chars)
    sig = signals(d.text, model=model, limit=limit)
    return AttrDict(doc_id=d.doc_id, title=d.title, method=sig.method, counts=sig.counts, ents=sig.ents)

## Schemas: the shape of an extraction

The shapes worth having ready, and `dyn_schema` for the ones that are not — a dataclass built from
`'vendor:str, total:float, items:dicts'` at the moment you ask for it.

In [ ]:
#| export
@dataclass
class Invoice:
    """An invoice, purchase order or quotation.

    items: one entry per line, each {description, qty, unit_price, amount}.
    Amounts are bare numbers with the currency in `currency`; dates are ISO (2024-03-01)."""
    number:str = ''
    date:str = ''
    due_date:str = ''
    vendor:str = ''
    vendor_tax_id:str = ''
    bill_to:str = ''
    ship_to:str = ''
    currency:str = ''
    subtotal:float = 0
    tax:float = 0
    total:float = 0
    payment_terms:str = ''
    items:list[dict] = None

@dataclass
class Receipt:
    """A receipt for a completed payment.

    items: one entry per line, each {description, qty, amount}. `paid_with` is the tender or the
    last digits of the card. Dates are ISO (2024-03-01)."""
    merchant:str = ''
    date:str = ''
    transaction_id:str = ''
    currency:str = ''
    subtotal:float = 0
    tax:float = 0
    total:float = 0
    paid_with:str = ''
    items:list[dict] = None

@dataclass
class Catalogue:
    """A product catalogue, price list or listing page.

    products: one entry per product, each {sku, name, category, price, currency, availability, url}.
    Prices are bare numbers."""
    name:str = ''
    vendor:str = ''
    currency:str = ''
    updated:str = ''
    n_products:int = 0
    products:list[dict] = None

@dataclass
class Contract:
    """An agreement between parties.

    parties: the legal names. obligations: what each side must do. Dates are ISO (2024-03-01)."""
    title:str = ''
    parties:list[str] = None
    effective_date:str = ''
    end_date:str = ''
    term:str = ''
    value:str = ''
    governing_law:str = ''
    termination:str = ''
    obligations:list[str] = None

@dataclass
class Resume:
    """One person's CV.

    experience: one entry per role, each {employer, title, start, end, summary}.
    education: one entry per qualification, each {institution, qualification, year}."""
    name:str = ''
    email:str = ''
    phone:str = ''
    location:str = ''
    headline:str = ''
    years_experience:float = 0
    skills:list[str] = None
    experience:list[dict] = None
    education:list[dict] = None

@dataclass
class Paper:
    """An academic paper. authors: names in order. findings: the claims the paper actually makes."""
    title:str = ''
    authors:list[str] = None
    venue:str = ''
    year:str = ''
    doi:str = ''
    abstract:str = ''
    method:str = ''
    findings:list[str] = None
    limitations:list[str] = None

@dataclass
class MeetingNotes:
    """Notes from one meeting. actions: one entry per commitment, each {what, owner, due}."""
    title:str = ''
    date:str = ''
    attendees:list[str] = None
    topics:list[str] = None
    decisions:list[str] = None
    actions:list[dict] = None

@dataclass
class Summary:
    """What one document says, when no more specific shape fits.

    entities: the organisations, people and places it names. dates: ISO where the document allows."""
    title:str = ''
    doctype:str = ''
    about:str = ''
    key_points:list[str] = None
    entities:list[str] = None
    dates:list[str] = None
    numbers:list[str] = None
    open_questions:list[str] = None

# A purchase order and a quotation carry an invoice's fields under other names, so they share its
# schema rather than getting a near-duplicate of it.
SCHEMAS = dict(invoice=Invoice, purchase_order=Invoice, quote=Invoice, receipt=Receipt,
               catalogue=Catalogue, contract=Contract, resume=Resume, paper=Paper,
               meeting_notes=MeetingNotes, other=Summary)

FIELD_TYPES = {'str':str, 'text':str, 'int':int, 'float':float, 'number':float, 'num':float, 'bool':bool,
               'list':list, 'dict':dict, 'strs':list[str], 'dicts':list[dict],
               'list[str]':list[str], 'list[dict]':list[dict]}
_DFLT = {str: '', int: 0, float: 0.0, bool: False}

def dyn_schema(spec,                    # 'vendor:str, total:float, items:dicts', or {'vendor':'str'}, or ['vendor']
               name:str='Extracted',    # class name, which the model sees
               doc:str=None,            # class docstring, which the model also sees
) -> type:
    """A dataclass built at runtime from a field spec — the shape of an answer, named in one string.

    This is what makes a structured response *dynamic*: the caller describes the fields they want in
    the question itself, and the model is constrained to them, without anyone declaring a class
    first. Every field defaults, so a document that does not say something yields a blank rather
    than an exception. `list` fields default to `None` rather than `[]` deliberately: a
    `default_factory` sentinel is not JSON-serialisable, and it is the *schema* that is shipped to
    the model — `extract` turns the `None`s back into empty lists on the way out. A type name that is
    not in `FIELD_TYPES` becomes `str`, on the same reasoning: a field read as text is recoverable, an
    exception raised halfway through a batch of documents is not."""
    if isinstance(spec, str): spec = [p for p in re.split(r'[,\n]', spec) if p.strip()]
    if isinstance(spec, dict): spec = [f'{k}:{v}' for k, v in spec.items()]
    flds = []
    for p in spec:
        nm, _, ty = (p if isinstance(p, str) else ':'.join(p)).partition(':')
        t = FIELD_TYPES.get(ty.strip().lower() or 'str', str)
        flds.append((re.sub(r'\W', '_', nm.strip()), t, _DFLT.get(t, None)))
    if not flds: raise ValueError(f'no fields in schema spec {spec!r}')
    return make_dataclass(re.sub(r'\W', '', name) or 'Extracted', flds,
                          namespace=dict(__doc__=doc or 'The fields to pull out of the document.'))

def as_schema(spec, name:str='Extracted', doc:str=None) -> type:
    'Whatever names a shape, as a dataclass: a `SCHEMAS` key, a dataclass, or a `dyn_schema` spec.'
    if is_dataclass(spec): return spec
    if isinstance(spec, str) and spec.strip() in SCHEMAS: return SCHEMAS[spec.strip()]
    return dyn_schema(spec, name=name, doc=doc)

def _norm(obj, schema) -> dict:
    "A structured reply as a plain dict, with the `None` a list field defaults to turned back into `[]`."
    d = asdict(obj) if is_dataclass(obj) and not isinstance(obj, type) else dict(obj or {})
    lists = {f.name for f in fields(schema) if f.type is list or get_origin(f.type) is list}
    return {k: ([] if v is None and k in lists else v) for k, v in d.items()}

## Extraction

One document in, a dict of fields out. With no `schema`, the document is categorised first and the
shape follows from what it turned out to be.

In [ ]:
#| export
EXTRACT_SP = """You pull structured fields out of one document.

Rules:
- Copy what the document states. Do not compute, convert, round or tidy a value.
- A field the document does not state stays empty. An invented number is worse than a blank one.
- Numbers are bare: 1240.50, not "$1,240.50". The currency belongs in its own field.
- Dates are ISO: 2024-03-01.
- A list field takes every entry the document has, in the order it has them."""

@patch
def extract(self:Vault,
            ref:str,               # doc_id, source, title substring, or a path on disk
            schema:str=None,       # a SCHEMAS key, a dataclass, or a 'field:type, …' spec; None -> from the doctype
            model:str=None,        # a MODELS alias or a full id
            runtime:str=None,      # 'litert' | 'mlx' | 'llama' | 'remote'
            max_chars:int=12000,   # chars of the document the model sees
            sp:str=EXTRACT_SP,     # system prompt for the extraction
            save:bool=False,       # write the fields into the document's meta
            llm:bool=None,         # the LLM leg of the categorisation, when picking the schema
) -> AttrDict:
    """Pull structured fields out of one document — an invoice's totals, a catalogue's products.

    With no `schema`, the document is categorised first and the shape follows from what it turned
    out to be, which is the whole point of typing a corpus: point this at a folder of mixed
    paperwork and each document is read against the shape that fits it. The model is constrained to
    the schema by rishi (a forced tool call on the hosted and LiteRT backends, a grammar on
    llama.cpp, a parsed JSON reply on MLX), so what comes back is a dict with the fields you asked
    for, not prose about them."""
    d = self.document(ref, max_chars=max_chars)
    if not (d.text or '').strip():
        return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=None, schema=None,
                        fields={}, skipped='no text to extract from')
    dt, cat = None, None
    if schema is None:
        cat = self.categorize(d, model=model, llm=llm, save=save)
        dt = cat.doctype
        sch = SCHEMAS.get(dt, Summary)
    else: sch = as_schema(schema)
    prompt = (f'{d.title}\n(source: {d.source})\n\n{d.text}\n\n---\n\n'
              f'Pull the fields of `{sch.__name__}` out of the document above.')
    ch = self.chat(model=model, runtime=runtime)
    flds = _norm(ch.structured(prompt, sch, sp=sp), sch)
    mid, rt = resolve_model(model, runtime)
    if save and d.doc_id: self.set_meta(d.doc_id, extracted=flds, extracted_as=sch.__name__)
    return AttrDict(doc_id=d.doc_id, title=d.title, source=d.source, doctype=dt, schema=sch.__name__,
                    fields=flds, chars=len(d.text), truncated=d.truncated, model=mid, runtime=rt,
                    categorized=cat, usage=getattr(ch, 'use', None))

@patch
def extract_all(self:Vault,
                doctype:str=None,   # only documents already categorised as this
                kind:str=None,      # only these KINDS
                schema:str=None,    # one shape for all of them; None -> each document's own doctype
                limit:int=None,     # stop after this many
                **kw                # forwarded to extract (model=, max_chars=, save=, sp=)
) -> AttrDict:
    """Extract from many documents at once — a folder of invoices as one table.

    `rows` is the point: every result flattened to `doc_id` and `doc_title` plus the schema's own
    fields, which is a dataframe, a CSV or a spreadsheet away from being useful. The identity columns
    carry the `doc_` prefix because `title` is a field of half the schemas — a document's own title
    and the vault's title for it are not always the same string, and neither should shadow the other.
    A document that yields nothing is recorded in `errors` and the run continues."""
    docs = self.of_type(doctype, kind) if doctype else self.sources(kind)
    rows, errs = L(), L()
    for r in docs[:limit]:
        try:
            e = self.extract(r['id'], schema=schema, **kw)
            if e.get('skipped'): errs.append(dict(doc_id=r['id'], title=r['title'], error=e.skipped))
            else: rows.append(dict({'doc_id': e.doc_id, 'doc_title': e.title, 'schema': e.schema},
                                   **e.fields))
        except Exception as ex:
            errs.append(dict(doc_id=r['id'], title=r['title'], error=f'{type(ex).__name__}: {str(ex)[:200]}'))
    return AttrDict(n=len(rows), doctype=doctype, schema=schema, rows=rows, errors=errs)

## Asking over one whole document

`ask` retrieves sections from everywhere. `ask_doc` starts from a document you have already chosen,
reads *all* of it, and adds the rest of the vault as context — and answers as prose or as a shape
you name.

In [ ]:
#| export
@patch
def ask_doc(self:Vault,
            ref:str,              # doc_id, source, title substring, or a path on disk
            question:str,         # what you want to know about it
            schema:str=None,      # answer as this shape instead of prose: a SCHEMAS key or a 'field:type, …' spec
            model:str=None,       # a MODELS alias or a full id
            runtime:str=None,     # 'litert' | 'mlx' | 'llama' | 'remote'
            max_chars:int=12000,  # chars of the document the model sees
            related:int=3,        # sections from the *rest* of the vault to add as context
            sp:str=VAULT_SP,      # system prompt
            fresh:bool=True,      # start a new conversation rather than continuing the last
) -> AttrDict:
    """Answer a question about one whole document — with the rest of the vault as context.

    `ask` retrieves sections from everywhere and answers across them; this starts from a document
    you have already chosen and reads all of it, which is what you want when the question is about
    *this* file: a markdown page, a source file, a contract. The document is section `[1]` and the
    `related` sections from other documents follow it, so the citation contract is the one `ask`
    already keeps — `[n]` in the answer resolves through `cited` to something you can `read()`.

    Pass `schema=` and the reply is a dict of the fields you named instead of prose: the same
    question, answered as data. The document need not be in the vault at all — a path on disk is
    read straight off it, so a file can be asked about before it is ever ingested."""
    from rishi.core import resp_text
    d = self.document(ref, max_chars=max_chars)
    res = L(AttrDict(node_id=f'{d.doc_id}#0' if d.doc_id else '', title=d.title, doc_id=d.doc_id,
                     breadcrumb=d.title, filename=d.source, text=d.text, pages=None))
    if related:
        for s in self.sections(question, limit=related*2):
            if s['node_id'].split('#')[0] == (d.doc_id or '~') or len(res) > related: continue
            res.append(AttrDict(node_id=s['node_id'], title=s['title'], doc_id=s['node_id'].split('#')[0],
                                breadcrumb=s['breadcrumb'], filename=None, pages=s['pages'],
                                text='\n\n'.join(s['snippets'])))
    parts = [f"[{i}] {r.breadcrumb}\n(source: {r.filename or r.doc_id or 'on disk'})\n\n{r.text}"
             for i, r in enumerate(res, 1)]
    prompt = ('\n\n---\n\n'.join(parts) + f'\n\n---\n\n[1] is the document being asked about; the rest '
              f'is context from elsewhere in the vault.\n\nQuestion: {question}')
    ch, mid_rt = self.chat(model=model, runtime=runtime, sp=sp), resolve_model(model, runtime)
    if fresh: ch.hist = []
    out = AttrDict(question=question, doc_id=d.doc_id, title=d.title, source=d.source, origin=d.origin,
                   chars=len(d.text), truncated=d.truncated, context=res.attrgot('breadcrumb'),
                   model=mid_rt[0], runtime=mid_rt[1], answer=None, cited=L(), schema=None,
                   fields=None, thinking='')
    if schema is not None:
        sch = as_schema(schema, name='Answer', doc=f'The answer to: {question}')
        out.schema, out.fields = sch.__name__, _norm(ch.structured(prompt, sch, sp=sp), sch)
    else:
        out.answer, out.thinking = split_reasoning(resp_text(ch(prompt)))
        out.answer, out.cited = out.answer.strip(), cited(out.answer, res)
    out.usage = getattr(ch, 'use', None)
    return out

## Try it

Two documents, one of each kind, and nothing here needs a model yet — the cue table, the signals and
the schema machinery are the legs that run everywhere.

In [ ]:
from vishalakshi import Vault

INVOICE = '''# INVOICE

Invoice No: ACM-2024-0117
Date: 2024-03-01
Payment terms: Net 30

Bill to: Contoso GmbH, Berlin
From: Acme Supplies Ltd

## Line items

| Description | Qty | Unit price | Amount |
|---|---|---|---|
| Widget, steel | 12 | $8.50 | $102.00 |
| Gasket, nitrile | 40 | $1.20 | $48.00 |

Subtotal: $150.00
VAT (20%): $30.00
Total due: $180.00
'''

CATALOGUE = '''# Spring price list

## Widget, steel

SKU: WS-100. Price: $8.50 per unit. In stock.
Specifications: 40mm, zinc plated. Dimensions 40x12x4mm.

## Gasket, nitrile

SKU: GN-220. Price: $1.20 per unit. Out of stock.
Add to cart to be notified.
'''

v = Vault(':memory:')
v.add(INVOICE, 'Acme invoice ACM-2024-0117', source='/inbox/acme-0117.md')
v.add(CATALOGUE, 'Spring price list', source='/inbox/spring-list.md')
v.doctypes()

In [ ]:
r = v.categorize('Acme invoice ACM-2024-0117', llm=False)
r.doctype, r.score, r.decisive, r.by

In [ ]:
test_eq(r.doctype, 'invoice')
assert r.decisive, r.scores          # the cues are sure enough that no model is needed
test_eq(v.doc(r.doc_id)['meta']['doctype'], 'invoice')   # written back into the vault
test_eq(v.categorize('Spring price list', llm=False).doctype, 'catalogue')
test_eq(v.doctypes(), {'invoice': 1, 'catalogue': 1})
test_eq(v.of_type('invoice').attrgot('title'), ['Acme invoice ACM-2024-0117'])

In [ ]:
#| hide
# the signals are the evidence, so they must be found whether or not spaCy is installed
sig = signals(INVOICE)
assert sig.method in ('regex', 'spacy+regex')
for k in ('money', 'date', 'ref', 'tax', 'percent', 'table', 'heading'): assert sig.counts.get(k), k
test_eq(signals(INVOICE, ner=False).method, 'regex')

# a score is a fraction, never a count: the same evidence twice over must not score higher
test_eq(cue_scores(INVOICE)['invoice'], cue_scores(INVOICE + INVOICE)['invoice'])
# and what acquired a document is evidence about what it is
test_eq(guess_type('Um, so, welcome back. 00:12 speaker 1: right.', kind='youtube').doctype, 'transcript')

# a document with nothing to go on says so rather than picking a type at random
g = guess_type('The cat sat on the mat.')
assert not g.decisive and g.score < MIN_SCORE, g
test_eq(v.categorize(AttrDict(doc_id=None, title='t', kind=None, text='   '),
                     llm=False).skipped, 'no text to judge')

In [ ]:
#| hide
# a dynamic schema is the point of `schema=` — the caller names the shape in one string
S = as_schema('vendor:str, total:float, paid:bool, items:dicts')
test_eq([f.name for f in fields(S)], ['vendor', 'total', 'paid', 'items'])
test_eq(asdict(S()), dict(vendor='', total=0.0, paid=False, items=None))
test_eq(_norm(S(), S)['items'], [])            # a list field comes back empty, not None
test_eq(as_schema('invoice'), Invoice)         # a registry key
test_eq(as_schema(Invoice), Invoice)           # a dataclass passes through
test_eq(as_schema({'sku': 'str'}).__name__, 'Extracted')
test_eq([f.name for f in fields(as_schema(['a', 'b']))], ['a', 'b'])
test_fail(lambda: as_schema(''), contains='no fields')

# every schema shipped to a model must be JSON-serialisable, which is why no field uses
# `default_factory` — its sentinel is not
from fastcore.funccall import get_schema
for nm, s in SCHEMAS.items(): assert json.dumps(get_schema(s)['input_schema']), nm

The legs above are the ones that run everywhere. The two that need a model — the label when the
cues cannot decide, and every structured extraction — are worth exercising too, and what is worth
testing about them is the plumbing rather than the weights: the prompt a whole document becomes, the
citation contract, and the shape that comes back. So the model is stubbed.

In [ ]:
#| hide
class FakeChat:
    'Enough of a `rishi.Chat` to exercise the three calls this module makes of one.'
    use = None
    def __init__(self): self.hist, self.seen = [], ''
    def __call__(self, prompt, **kw):
        self.seen = prompt
        return 'Total due is 180.00 [1]; the price list [2] agrees on the unit price.'
    def classify(self, text, labels, sp=None): self.seen = text; return 'receipt'
    def structured(self, prompt, schema, sp=None):
        self.seen, self.schema = prompt, schema
        return schema(**{f.name: 'stubbed' for f in fields(schema) if f.type is str})

@patch
def chat(self:Vault, model:str=None, runtime:str=None, sp:str=None, **kw): return self._fake
v._fake = FakeChat()

In [ ]:
#| hide
# llm=True must override the cues, and record that it did
c = v.categorize('Acme invoice', llm=True, save=False)
test_eq((c.doctype, c.by.startswith('llm')), ('receipt', True))
assert 'Total due' in v._fake.seen                      # the model saw the document, not a chunk of it
# a reply that is not one of the labels leaves the cue verdict standing, and says so
v._fake.classify = lambda text, labels, sp=None: 'a kind of ledger, I think'
c = v.categorize('Acme invoice', llm=True, save=False)
test_eq(c.doctype, 'invoice')
assert 'not a label' in c.by, c.by
# and `llm=None` means "a model if one can be had": no rishi, no weights, no network — the cue
# verdict stands rather than the run dying
def boom(text, labels, sp=None): raise ImportError('no rishi here')
v._fake.classify = boom
mat = AttrDict(doc_id=None, title='untitled', kind=None, text='The cat sat on the mat.')
assert 'no model available (ImportError)' in v.categorize(mat, llm=None, save=False).by
test_fail(lambda: v.categorize(mat, llm=True, save=False), contains='no rishi here')

In [ ]:
#| hide
# extraction: the shape is named, the whole document is what gets read, and a list field that the
# model left unset comes back as [] rather than None
e = v.extract('Acme invoice', schema='invoice')
test_eq((e.schema, e.fields['number'], e.fields['items']), ('Invoice', 'stubbed', []))
assert 'Gasket, nitrile' in v._fake.seen and str(v._fake.schema.__name__) == 'Invoice'
# no schema: the doctype picks it
test_eq(v.extract('Spring price list').schema, 'Catalogue')
test_eq(v.extract('Acme invoice', schema='po_number:str, total:float').fields, dict(po_number='stubbed', total=0.0))
# across the corpus, one row per document — the table a folder of paperwork becomes
b = v.extract_all(doctype='invoice')
test_eq((b.n, len(b.errors)), (1, 0))
test_eq(b.rows[0]['doc_title'], 'Acme invoice ACM-2024-0117')
# `title` is a field of Paper, Contract and Catalogue: the identity columns must not collide with it
test_eq(sorted(v.extract_all(kind='file', schema=Paper).rows[0])[:3], ['abstract', 'authors', 'doc_id'])
# and a document with nothing in it is an error, not an empty row
v.add('   ', 'empty one', source='/inbox/empty.md')
e = v.extract('/inbox/empty.md')
test_eq((e.skipped, e.fields), ('no text to extract from', {}))

In [ ]:
#| hide
# ask_doc: the document is section [1] and the rest of the vault follows it, so `ask`'s citation
# contract holds — [n] resolves to something read() can open
a = v.ask_doc('Acme invoice', 'what is the total?', related=2)
test_eq(a.cited.attrgot('node_id')[0], f'{a.doc_id}#0')
assert '[1] Acme invoice' in v._fake.seen and 'Question: what is the total?' in v._fake.seen
assert 'VAT (20%)' in v._fake.seen, 'the whole document must reach the model, not a retrieved chunk'
assert all('Acme' not in c for c in a.context[1:]), 'the context leg must exclude the document itself'
# the same question, answered as data instead of prose
d = v.ask_doc('Acme invoice', 'what is the total?', schema='total:float, currency:str')
test_eq((sorted(d.fields), d.answer), (['currency', 'total'], None))

# and a file the vault has never seen, straight off disk
from fastcore.all import Path
from tempfile import mkdtemp
p = Path(mkdtemp())/'notes.md'
p.write_text('# Standup\n\nAttendees: Ana, Bo. Action items: Ana to ship the parser.')
test_eq(v.document(p).origin, 'disk')
test_eq(v.ask_doc(str(p), 'who is shipping the parser?').origin, 'disk')
test_eq(v.categorize(str(p), llm=False, save=False).doctype, 'meeting_notes')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()